In [6]:
pip install thop torchinfo

Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import time
import os
from thop import profile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_model_stats(model, input_size=(1, 3, 224, 224), repetitions=100, warmup=10):
    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(input_size).to(device)

    # -----------------------
    # Parameters
    # -----------------------
    params = sum(p.numel() for p in model.parameters()) / 1e6

    # -----------------------
    # FLOPs
    # -----------------------
    try:
        flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
        flops = flops / 1e9
    except:
        flops = 0.0

    # -----------------------
    # Model Size
    # -----------------------
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # -----------------------
    # Inference Time (FIXED)
    # -----------------------
    timings = []

    with torch.no_grad():
        # Warm-up
        for _ in range(warmup):
            _ = model(dummy_input)

        if device.type == "cuda":
            starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

            for _ in range(repetitions):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))
        else:
            for _ in range(repetitions):
                start = time.time()
                _ = model(dummy_input)
                end = time.time()
                timings.append((end - start) * 1000)

    inf_time = sum(timings) / len(timings)

    # -----------------------
    # GPU Memory
    # -----------------------
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(dummy_input)
        mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        mem = 0.0

    return {
        "Params(M)": round(params, 2),
        "FLOPs(G)": round(flops, 2),
        "Size(MB)": round(size_mb, 2),
        "Inf Time(ms)": round(inf_time, 2),
        "GPU Mem(GB)": round(mem, 2)
    }

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class TransUNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        resnet = models.resnet50(weights=None)

        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4

        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        self.dec4 = DecoderBlock(2048, 1024)
        self.dec3 = DecoderBlock(1024, 512)
        self.dec2 = DecoderBlock(512, 256)
        self.dec1 = DecoderBlock(256, 64)

        self.final = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        x = self.enc5(x)

        x = self.up(self.dec4(x))
        x = self.up(self.dec3(x))
        x = self.up(self.dec2(x))
        x = self.up(self.dec1(x))

        return self.final(x)

In [9]:
model = TransUNet(num_classes=2)
stats = compute_model_stats(model, (1, 3, 224, 224))
print(stats)

{'Params(M)': 60.86, 'FLOPs(G)': 8.9, 'Size(MB)': 232.51, 'Inf Time(ms)': 11.57, 'GPU Mem(GB)': 0.59}
